# Lab 2 — Structural Design & Multi-Physics Analysis

**Physics Workspace M³ — Composite Wind Turbine Design**

Domínios conforme INSTRUCTIONS.md KDI:
- **Mecânica**: tensões, deformações, flambagem, fadiga
- **Fluidos**: aerodinâmica de pás, BEM theory, perfil de vento
- **Termodinâmica**: processos de secagem/cura, eficiência
- **Energia**: potência eólica, conversão, armazenamento
- **Eletricidade**: gerador PMSG, inversor, perdas

Pipeline: Geometria → Cargas → Tensões → Falha → Otimização

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../modules'))
import numpy as np
import matplotlib.pyplot as plt

from structural_analysis import (
    BeamSection, CompositeLaminate, PlyProperties,
    von_mises_stress, principal_stresses,
    tsai_wu_failure, max_stress_failure, safety_factor
)
from fluid_dynamics import (
    wind_profile, bem_theory, turbine_power,
    Airfoil, reynolds_number
)
from thermodynamics import DryingProcess, carnot_efficiency, exergy
from electromechanical import PMSG, BatteryStorage, power_conversion_chain

print('All Lab 2 modules loaded')

In [ ]:
# ---- Step 1: Wind Environment (Macro) ----
v_ref = 6.0  # m/s at 10m
v_hub = wind_profile(12, v_ref, z_ref_m=10, terrain='open')
print(f'Wind at 10m: {v_ref} m/s')
print(f'Wind at hub (12m): {v_hub:.2f} m/s')

# Reynolds number for blade root
chord = 0.3  # m
Re = reynolds_number(v_hub, chord)
print(f'Reynolds: {Re:.0f} — flow: {"turbulent" if Re > 40000 else "laminar"}')

In [ ]:
# ---- Step 2: Blade Aerodynamics (BEM) ----
R = 1.5  # rotor radius (m)
rpm = 200

bem = bem_theory(v_hub, rpm, R)
print('=== BEM Analysis (r/R=0.75) ===')
for k, v in bem.items():
    print(f'  {k}: {v}')

power = turbine_power(v_hub, R, Cp=0.35)
print(f'\n=== Turbine Power ===')
print(f'  Rotor area: {power["rotor_area_m2"]} m²')
print(f'  Wind power: {power["wind_power_W"]/1000:.2f} kW')
print(f'  Output: {power["power_W"]/1000:.2f} kW at Cp={power["Cp"]}')

In [ ]:
# ---- Step 3: Composite Blade Structure ----
laminate = CompositeLaminate()
laminate.add_ply(PlyProperties(E1_GPa=8, E2_GPa=4, G12_GPa=2, nu12=0.3,
                               thickness_mm=0.5, orientation_deg=0))   # coating
laminate.add_ply(PlyProperties(E1_GPa=3, E2_GPa=2, G12_GPa=1, nu12=0.35,
                               thickness_mm=4.0, orientation_deg=45))  # core
laminate.add_ply(PlyProperties(E1_GPa=8, E2_GPa=4, G12_GPa=2, nu12=0.3,
                               thickness_mm=0.5, orientation_deg=0))   # coating

ABD = laminate.ABD_matrix()
print('=== Laminate ABD Matrix ===')
print('A (extensional):'); print(np.round(ABD['A'], 1))
print('B (coupling):'); print(np.round(ABD['B'], 2))
print('D (bending):'); print(np.round(ABD['D'], 1))
print(f'\nB-matrix norm (coupling): {np.linalg.norm(ABD["B"]):.2f}')
print(f'{"BALANCED" if np.linalg.norm(ABD["B"]) < 0.1 else "UNBALANCED"} laminate')

In [ ]:
# ---- Step 4: Structural Analysis ----
beam = BeamSection(width_mm=50, height_mm=10, E_modulus_GPa=3.5,
                   length_mm=1500, density_kgm3=950)

# Maximum bending load at blade root
thrust_N = bem['thrust_N']
moment_Nm = thrust_N * R  # cantilever root moment
sigma = beam.bending_stress(moment_Nm)
deflection = beam.deflection_point(thrust_N, 'end')
f_nat = beam.natural_frequency_Hz(mode=1)

print('=== Structural Analysis ===')
print(f'  Thrust: {thrust_N:.1f} N')
print(f'  Root moment: {moment_Nm:.1f} Nm')
print(f'  Max bending stress: {sigma:.1f} MPa')
print(f'  Tip deflection: {deflection:.1f} mm')
print(f'  1st natural frequency: {f_nat:.1f} Hz')

# Failure criteria
fw = tsai_wu_failure(sigma, sigma*0.1, sigma*0.05,
                     Xt=40, Xc=25, Yt=10, Yc=15, S=8)
fs = max_stress_failure(sigma, sigma*0.1, sigma*0.05,
                        Xt=40, Xc=25, Yt=10, Yc=15, S=8)
print(f'\n  Tsai-Wu: FI={fw["failure_index"]} → {fw["status"]}')
print(f'  Max Stress: FI={fs["failure_index"]} → {fs["status"]}')
print(f'  Safety Factor: {safety_factor(40, sigma):.1f}')

In [ ]:
# ---- Step 5: Generator & Power Conversion ----
gen = PMSG(power_rated_W=1500, voltage_V=48, poles=8, rpm_rated=400)
print('=== PMSG Generator ===')
for k, v in gen.summary().items():
    print(f'  {k}: {v}')

# Power conversion
P_turbine = power['power_W']
conv = power_conversion_chain(P_turbine)
print(f'\n=== Power Conversion ===')
for k, v in conv.items():
    print(f'  {k}: {v}')

# Storage
bat = BatteryStorage(capacity_Wh=2000, voltage_nominal_V=48)
print(f'\n=== Storage ===')
for k, v in bat.summary().items():
    print(f'  {k}: {v}')

In [ ]:
# ---- Step 6: Manufacturing Thermodynamics ----
dry = DryingProcess(mass_kg=2.0, water_content_pct=50,
                    drying_temp_C=60, initial_temp_C=25)
print('=== Drying Process ===')
print(f'  Heat energy: {dry.energy_to_heat_J()/1e3:.0f} kJ')
print(f'  Evaporation: {dry.energy_to_evaporate_J()/1e3:.0f} kJ')
print(f'  Total: {dry.total_energy_MJ():.1f} MJ')

# Exergy analysis
ex = exergy(dry.total_energy_MJ() * 1e6, T_source_K=dry.drying_temp_C + 273)
print(f'  Exergy fraction: {ex["exergy_fraction_pct"]}%')

In [ ]:
# ---- Step 7: Visualization ----
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Wind profile
ax = axes[0,0]
heights = np.linspace(1, 30, 50)
speeds = [wind_profile(h, v_ref) for h in heights]
ax.plot(speeds, heights, 'b-', linewidth=2)
ax.axhline(y=12, color='r', linestyle='--', alpha=0.5, label='hub height')
ax.set_xlabel('Wind speed (m/s)'); ax.set_ylabel('Height (m)')
ax.set_title('Wind Profile'); ax.grid(True); ax.legend()

# BEM power vs TSR
ax = axes[0,1]
tsrs = np.linspace(1, 10, 20)
powers = []
for tsr in tsrs:
    bem_rpm = tsr * v_hub / R * 60 / (2*np.pi)
    p = turbine_power(v_hub, R, Cp=0.35)
    powers.append(p['power_W'])
ax.plot(tsrs, np.array(powers)/1000, 'g-')
ax.set_xlabel('TSR'); ax.set_ylabel('Power (kW)')
ax.set_title('Power vs TSR'); ax.grid(True)

# Stress distribution
ax = axes[0,2]
span = np.linspace(0, R, 20)
stresses = []
for s in span:
    b = BeamSection(length_mm=s*1000)
    m = thrust_N * s
    stresses.append(b.bending_stress(m) if m > 0 else 0)
ax.plot(span*1000, stresses, 'r-', linewidth=2)
ax.set_xlabel('Span (mm)'); ax.set_ylabel('Stress (MPa)')
ax.set_title('Blade Stress Distribution'); ax.grid(True)

# Generator efficiency
ax = axes[1,0]
loads = np.linspace(100, 3000, 30)
effs = [gen.efficiency(l) for l in loads]
ax.plot(loads, effs, 'm-', linewidth=2)
ax.set_xlabel('Output Power (W)'); ax.set_ylabel('Efficiency (%)')
ax.set_title('Generator Efficiency'); ax.grid(True)

# Failure criteria comparison
ax = axes[1,1]
stresses_test = np.linspace(1, 60, 30)
fi_tsai = [tsai_wu_failure(s, s*0.1, s*0.05, 40, 25, 10, 15, 8)['failure_index'] for s in stresses_test]
fi_max = [max_stress_failure(s, s*0.1, s*0.05, 40, 25, 10, 15, 8)['failure_index'] for s in stresses_test]
ax.plot(stresses_test, fi_tsai, 'b-', label='Tsai-Wu')
ax.plot(stresses_test, fi_max, 'r--', label='Max Stress')
ax.axhline(y=1, color='k', linestyle=':', label='Failure')
ax.set_xlabel('Applied Stress (MPa)'); ax.set_ylabel('Failure Index')
ax.set_title('Failure Criteria'); ax.grid(True); ax.legend()


# Energy flow
ax = axes[1,2]
ax.axis('off')
labels = ['Wind', 'Rotor', 'Generator', 'Inverter', 'Load']
values = [
    power['wind_power_W'],
    power['power_W'],
    power['power_W'] * gen.efficiency(P_turbine)/100,
    power['power_W'] * gen.efficiency(P_turbine)/100 * conv['inverter_efficiency_pct']/100,
    power['power_W'] * gen.efficiency(P_turbine)/100 * conv['total_efficiency_pct']/100,
]
ax.barh(labels, np.array(values)/1000, color=['skyblue','green','orange','purple','blue'])
ax.set_xlabel('Power (kW)')
ax.set_title('Energy Conversion Chain')

plt.tight_layout()
plt.show()

print('\n=== Lab 2 Complete ===')
print(f'Turbine: {P_turbine/1000:.2f} kW at {v_hub:.1f} m/s')
print(f'Stress: {sigma:.1f} MPa, SF: {safety_factor(40, sigma):.1f}')
print(f'Gen efficiency: {gen.efficiency(P_turbine)}%')